# Сохранение RuREBus на Google Drive

Ноутбук подключает Google Drive и сохраняет репозиторий RuREBus в:

`MyDrive/NER_RuREBus_project/rurebus_data/RuREBus/`

Ячейки можно запускать повторно: уже загруженный датасет не скачивается заново.

In [ ]:
# 1. Подключаем Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. Задаём и проверяем пути
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/NER_RuREBus_project")
DATA_DIR = PROJECT_DIR / "rurebus_data"
RUREBUS_DIR = DATA_DIR / "RuREBus"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Папка проекта:  {PROJECT_DIR}")
print(f"Папка данных:   {DATA_DIR}")
print(f"Папка RuREBus: {RUREBUS_DIR}")

In [ ]:
# 3. Загружаем RuREBus с GitHub
import subprocess

REPOSITORY_URL = "https://github.com/dialogue-evaluation/RuREBus.git"

if (RUREBUS_DIR / ".git").is_dir():
    print("RuREBus уже загружен. Повторное скачивание не требуется.")
elif RUREBUS_DIR.exists():
    raise FileExistsError(
        f"Папка {RUREBUS_DIR} уже существует, но не является Git-репозиторием. "
        "Проверьте её содержимое или выберите другое имя папки."
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(RUREBUS_DIR)],
        check=True,
    )
    print("RuREBus успешно сохранён на Google Drive.")

In [ ]:
# 4. Проверяем наличие основных папок и считаем файлы
required_directories = ["train_data", "test_data"]

for directory_name in required_directories:
    directory = RUREBUS_DIR / directory_name
    if not directory.is_dir():
        raise FileNotFoundError(f"Не найдена ожидаемая папка: {directory}")

    txt_count = sum(1 for _ in directory.rglob("*.txt"))
    ann_count = sum(1 for _ in directory.rglob("*.ann"))
    print(f"{directory_name}: TXT-файлов — {txt_count}, ANN-файлов — {ann_count}")

print("\nПроверка завершена: основные папки датасета доступны.")

In [ ]:
# 5. Показываем объём датасета и свободное место на Google Drive
import shutil

dataset_size_bytes = sum(
    path.stat().st_size for path in RUREBUS_DIR.rglob("*") if path.is_file()
)
drive_total, drive_used, drive_free = shutil.disk_usage("/content/drive")

print(f"Размер RuREBus:       {dataset_size_bytes / 1024**2:.2f} МБ")
print(f"Свободно на Drive:   {drive_free / 1024**3:.2f} ГБ")
print(f"Итоговый путь:       {RUREBUS_DIR}")